# TP2 - Visualisation avancée avec Parametric UMAP


## Pre-execution

### Configuration TensorFlow

In [36]:
import tensorflow as tf

### Imports

In [37]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.neighbors import KNeighborsClassifier
import umap
from umap import ParametricUMAP


## Data Preparation

### Load data


In [38]:
print("1. Chargement des données...")
columns = [
    "Elevation",
    "Aspect",
    "Slope",
    "Horizontal_Distance_To_Hydrology",
    "Vertical_Distance_To_Hydrology",
    "Horizontal_Distance_To_Roadways",
    "Hillshade_9am",
    "Hillshade_Noon",
    "Hillshade_3pm",
    "Horizontal_Distance_To_Fire_Points",
    "Wilderness_Area1",
    *[f"Soil_Type_{i}" for i in range(1, 40)],
    "Cover_Type"
]

1. Chargement des données...


In [39]:
data = pd.read_csv('https://archive.ics.uci.edu/ml/machine-learning-databases/covtype/covtype.data.gz',
                   header=None, names=columns)
data = data.sample(n=20000, random_state=42)

### Prepare data

In [40]:
# Features et labels
X = data.drop('Cover_Type', axis=1).astype('float32')
y = data['Cover_Type'].astype('int32')

# Conversion en tensors TensorFlow
X_tensor = tf.convert_to_tensor(X, dtype=tf.float32)
y_tensor = tf.convert_to_tensor(y, dtype=tf.int32)

# Normalisation des features
normalizer = tf.keras.layers.Normalization(axis=-1)
normalizer.adapt(X_tensor)
X_normalized = normalizer(X_tensor)

# Conversion en tableaux NumPy pour la compatibilité avec train_test_split
X_normalized_np = X_normalized.numpy()
y_np = y_tensor.numpy()

# Split des données
X_train, X_test, y_train, y_test = train_test_split(X_normalized_np, y_np, test_size=0.2, random_state=42)


## Parametric UMAP Execution


In [ ]:

print("\n2. Exécution de Parametric UMAP avec GPU...")
parametric_umap = ParametricUMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    random_state=42
)
X_parametric_umap_train = parametric_umap.fit_transform(X_train)
X_parametric_umap_test = parametric_umap.transform(X_test)
print("Parametric UMAP terminé avec succès.")



2. Exécution de Parametric UMAP avec GPU...


C:\Users\alaac\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Epoch 1/10


C:\Users\alaac\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\layer.py:391: UserWarning: `build()` was called on layer 'umap_model', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


1939/1939 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - loss: 0.1598
Epoch 2/10
1939/1939 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - loss: 0.1397
Epoch 3/10
1939/1939 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - loss: 0.1392
Epoch 4/10
1939/1939 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - loss: 0.1389
Epoch 5/10
1939/1939 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - loss: 0.1388
Epoch 6/10
1939/1939 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - loss: 0.1387
Epoch 7/10
 538/1939 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.1387


## Execution de UMAP classique


In [ ]:
print("\n2. Exécution de UMAP classique...")
umap_model = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    random_state=42
)
X_umap_train = umap_model.fit_transform(X_train)
X_umap_test = umap_model.transform(X_test)
print("UMAP classique terminé avec succès.")

## Execution de Parametric UMAP 


In [ ]:

## Parametric UMAP Execution
print("\n2. Exécution de Parametric UMAP avec GPU...")
parametric_umap = ParametricUMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    random_state=42
)
X_parametric_umap_train = parametric_umap.fit_transform(X_train)
X_parametric_umap_test = parametric_umap.transform(X_test)
print("Parametric UMAP terminé avec succès.")


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_normalized_np, y_np, test_size=0.2, random_state=42)

## Application de KNN sur les projections Parametric UMAP et UMAP classique

In [ ]:

## Modèles de classification
print("\n4. Évaluation des performances de classification...")

# KNN sur les projections Parametric UMAP
knn_parametric = KNeighborsClassifier(n_neighbors=5)
knn_parametric.fit(X_parametric_umap_train, y_train)
y_pred_parametric = knn_parametric.predict(X_parametric_umap_test)

# KNN sur les projections UMAP classique
knn_classic = KNeighborsClassifier(n_neighbors=5)
knn_classic.fit(X_umap_train, y_train)
y_pred_classic = knn_classic.predict(X_umap_test)



## Calcul des métriques


In [ ]:
print("\nMétriques pour Parametric UMAP :")
print(f"Accuracy : {accuracy_score(y_test, y_pred_parametric):.4f}")
print(f"F1 Score (macro) : {f1_score(y_test, y_pred_parametric, average='macro'):.4f}")
print("\nRapport de classification :")
print(classification_report(y_test, y_pred_parametric))

print("\nMétriques pour UMAP classique :")
print(f"Accuracy : {accuracy_score(y_test, y_pred_classic):.4f}")
print(f"F1 Score (macro) : {f1_score(y_test, y_pred_classic, average='macro'):.4f}")
print("\nRapport de classification :")
print(classification_report(y_test, y_pred_classic))



## Visualisation des résultats

In [ ]:
plt.figure(figsize=(15, 7))

# Parametric UMAP
plt.subplot(1, 2, 1)
plt.scatter(X_parametric_umap_train[:, 0], X_parametric_umap_train[:, 1], c=y_train, cmap='tab10', s=5)
plt.title("Projection Parametric UMAP")

# UMAP classique
plt.subplot(1, 2, 2)
plt.scatter(X_umap_train[:, 0], X_umap_train[:, 1], c=y_train, cmap='tab10', s=5)
plt.title("Projection UMAP classique")

plt.show()
